# 10 · DACON Stage 3 calibration — v4-A → competition labels

This notebook does **not** retrain the V-JEPA backbone. It freezes the completed
v4-A checkpoint and uses the tiny released Stage 3 example labels only to
calibrate the final categorical mapping.

Key safeguards:

- Public `frame_index` is used only to align the released example video.
- Hidden evaluation is treated as 10 Hz / decoded-frame = `sample_index`.
- Calibration is grouped by video with leave-one-video-out (LOVO) diagnostics.
- `metrics.py` remains unchanged.
- Both **LOVO-consensus** and **all-label-best** calibrations are exported; the
  consensus variant is the first-submission default because the released example
  is small and differs from hidden evaluation.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Initial Drive mount failed; forcing remount:", repr(exc))
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        print("WARNING: local repo has changes; git pull skipped.")
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)

COLAB_EXTRAS = [
    "timm==1.0.15",
    "fvcore==0.1.5.post20221221",
    "iopath==0.1.10",
    "yacs==0.1.8",
    "einops==0.8.1",
    "easydict==1.13",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.calibration import (
    Stage3Calibration,
    apply_calibration,
    evaluate_calibration,
    leave_one_video_out_calibration,
    save_calibration_report,
    validate_labels,
)
from blackbox_detection.stage3.dacon_inference import (
    extract_video_features,
    infer_public_frame_mapping,
)
from blackbox_detection.stage3.metrics import dacon_stage3_metrics
from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.utils.checkpoint import load_checkpoint

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_DATA_ROOT = Path("/content/dacon_stage3_public")
LOCAL_PRETRAINED_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

CAL_CFG_PATH = REPO / "configs" / "stage3" / "dacon_stage3_calibration_v1.yaml"
V4_CFG_PATH = REPO / "configs" / "stage3" / "vjepa21b_can_accel_v4a.yaml"
cal_cfg = yaml.safe_load(CAL_CFG_PATH.read_text(encoding="utf-8"))
v4_cfg = yaml.safe_load(V4_CFG_PATH.read_text(encoding="utf-8"))

RUN_NAME = cal_cfg["experiment"]["name"]
CAL_OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
CAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert_dacon_metric_contract()
print("repo      :", REPO)
print("cal config:", CAL_CFG_PATH)
print("v4 config :", V4_CFG_PATH)
print("output    :", CAL_OUTPUT_DIR)
print("DACON metric contract: PASS")


## 1. Locate the released public Stage 3 example

If an extracted Stage 3 folder is not found, the cell tries the configured
`Baseline.zip` paths and extracts **only Stage 3 members** to local Colab disk.

If your `Baseline.zip` is stored elsewhere, edit `BASELINE_ZIP_OVERRIDE`.


In [ ]:
BASELINE_ZIP_OVERRIDE = None  # e.g. Path("/content/drive/MyDrive/.../Baseline.zip")


def find_stage3_root(candidates):
    checked = []
    for candidate in candidates:
        p = Path(candidate)
        checked.append(p)
        if not p.exists():
            continue
        label_files = [p] if p.name.lower() == "labels.csv" else list(p.rglob("labels.csv"))
        for labels_path in label_files:
            parent = labels_path.parent
            if any(parent.rglob("*.mp4")) or any(parent.rglob("*.MP4")):
                return parent, labels_path
    return None, None


root_candidates = [Path(x) for x in cal_cfg["public_data"]["extracted_root_candidates"]]
stage3_root, labels_path = find_stage3_root(root_candidates)

if stage3_root is None:
    zip_candidates = []
    if BASELINE_ZIP_OVERRIDE is not None:
        zip_candidates.append(Path(BASELINE_ZIP_OVERRIDE))
    zip_candidates.extend(
        Path(x) for x in cal_cfg["public_data"]["baseline_zip_candidates"]
    )
    baseline_zip = next((p for p in zip_candidates if p.is_file()), None)
    if baseline_zip is None:
        raise FileNotFoundError(
            "Could not find extracted Stage 3 public data or Baseline.zip.\n"
            "Put Baseline.zip at one of:\n"
            + "\n".join(f"  - {p}" for p in zip_candidates)
            + "\nor set BASELINE_ZIP_OVERRIDE."
        )

    print("Extracting only Stage 3 public members from:", baseline_zip)
    with zipfile.ZipFile(baseline_zip) as zf:
        members = [
            name for name in zf.namelist()
            if "/stage3/" in name.lower()
            or name.lower().startswith("data/stage3/")
            or name.lower().startswith("stage3/")
        ]
        if not members:
            raise RuntimeError("Baseline.zip contains no recognizable Stage 3 members")
        for member in members:
            zf.extract(member, LOCAL_DATA_ROOT)

    stage3_root, labels_path = find_stage3_root([LOCAL_DATA_ROOT])

if stage3_root is None or labels_path is None:
    raise RuntimeError("Could not resolve public Stage 3 root after extraction")

labels = validate_labels(pd.read_csv(labels_path))

video_index = {}
for p in sorted([*stage3_root.rglob("*.mp4"), *stage3_root.rglob("*.MP4")]):
    video_index.setdefault(p.stem, p)

missing_videos = sorted(set(labels["ID"]) - set(video_index))
if missing_videos:
    raise FileNotFoundError(f"Missing public videos for IDs: {missing_videos}")

print("Stage 3 root:", stage3_root)
print("labels      :", labels_path)
print("rows / IDs  :", len(labels), labels["ID"].nunique())
print("class accel :", labels["accel_label"].value_counts().to_dict())
print("class steer :", labels["steer_label"].value_counts().to_dict())

mapping_rows = []
for video_id, group in labels.groupby("ID", sort=True):
    stride, offset = infer_public_frame_mapping(group)
    mapping_rows.append({"ID": video_id, "raw_stride": stride, "raw_offset": offset})
display(pd.DataFrame(mapping_rows))


## 2. Load v4-A best checkpoint

This reuses the exact v4-A model architecture and target normalization statistics.
The official V-JEPA checkpoint is staged locally because Drive FUSE is unreliable
for large repeated random reads.


In [ ]:
def is_usable(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= min_bytes
    except OSError:
        return False


def copy_to_local(source: Path, dest: Path, *, min_bytes: int = 1):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".tmp")
    tmp.unlink(missing_ok=True)
    with source.open("rb") as src, tmp.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    if tmp.stat().st_size < min_bytes:
        raise OSError(f"staged file too small: {tmp.stat().st_size}")
    os.replace(tmp, dest)


stats_path = MANIFEST_ROOT / "target_stats.json"
stats = json.loads(stats_path.read_text(encoding="utf-8"))

VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"
if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/facebookresearch/vjepa2.git", str(VJEPA_REPO)],
        check=True,
    )
subprocess.run(["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT], check=True)

VJEPA_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_DRIVE = PRETRAINED_ROOT / VJEPA_NAME
VJEPA_LOCAL = LOCAL_PRETRAINED_ROOT / VJEPA_NAME
if not is_usable(VJEPA_LOCAL, 1_000_000_000):
    if not is_usable(VJEPA_DRIVE, 1_000_000_000):
        raise FileNotFoundError(
            f"Official V-JEPA checkpoint not found on Drive: {VJEPA_DRIVE}"
        )
    copy_to_local(VJEPA_DRIVE, VJEPA_LOCAL, min_bytes=1_000_000_000)

V4_RUN = cal_cfg["experiment"]["source_model"]
V4_CKPT_DRIVE = OUTPUT_ROOT / V4_RUN / cal_cfg["experiment"]["source_checkpoint"]
V4_CKPT_LOCAL = LOCAL_PRETRAINED_ROOT / f"{V4_RUN}__best.pt"
if not is_usable(V4_CKPT_LOCAL, 1_000_000):
    if not is_usable(V4_CKPT_DRIVE, 1_000_000):
        raise FileNotFoundError(f"v4-A best checkpoint not found: {V4_CKPT_DRIVE}")
    copy_to_local(V4_CKPT_DRIVE, V4_CKPT_LOCAL, min_bytes=1_000_000)

from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.models import VJEPA21DenseCAN

dc = v4_cfg["data"]
mc = v4_cfg["model"]
fusion_cfg = dict(mc.get("accel_fusion") or {})

backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_LOCAL,
    num_frames=int(dc["clip_len"]),
    out_layers=tuple(mc["out_layers"]),
    freeze=True,
)
model = VJEPA21DenseCAN(
    backbone,
    freeze_backbone=True,
    feature_dim=int(mc["feature_dim"]),
    temporal_hidden=int(mc["temporal_hidden"]),
    temporal_layers=int(mc["temporal_layers"]),
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=bool(fusion_cfg.get("enabled", True)),
    accel_fusion_hidden=int(fusion_cfg.get("hidden", 64)),
    accel_fusion_gate_init=float(fusion_cfg.get("gate_init", 0.10)),
    accel_fusion_detach_ordinal_inputs=bool(
        fusion_cfg.get("detach_ordinal_inputs", True)
    ),
)

metadata = load_checkpoint(
    V4_CKPT_LOCAL,
    model=model,
    optimizer=None,
    scheduler=None,
    map_location="cpu",
    strict=True,
    restore_rng_state=False,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

thresholds = [float(x) for x in cal_cfg["calibration"]["ordinal_thresholds_mps2"]]
if thresholds != [float(x) for x in mc["accel_ordinal_thresholds_mps2"]]:
    raise ValueError("calibration and v4-A ordinal thresholds differ")

print("device          :", device)
print("v4 checkpoint   :", V4_CKPT_LOCAL)
print("checkpoint epoch:", metadata.get("epoch"))
print("ordinal thr     :", thresholds)


## 3. Extract dense public-video features

We infer **every 10-Hz sample**, not only the 50 labeled rows, so temporal
smoothing during calibration is well-defined. The tiny labels are merged only
after dense inference.

The public video is sampled by the exact integer mapping inferred from
`frame_index`; container FPS is never used.


In [ ]:
FEATURES_PATH = CAL_OUTPUT_DIR / "public_v4a_features.csv"
FORCE_REEXTRACT = False

if FEATURES_PATH.is_file() and not FORCE_REEXTRACT:
    features = pd.read_csv(FEATURES_PATH)
    print("Loaded cached features:", FEATURES_PATH, features.shape)
else:
    tables = []
    for video_id, group in labels.groupby("ID", sort=True):
        stride, offset = infer_public_frame_mapping(group)
        video_path = video_index[video_id]
        print(f"[{video_id}] {video_path.name} | public raw stride={stride}, offset={offset}")

        table = extract_video_features(
            model,
            video_path,
            video_id=video_id,
            target_stats=stats,
            ordinal_thresholds_mps2=thresholds,
            input_height=int(dc["input_height"]),
            input_width=int(dc["input_width"]),
            raw_stride=stride,
            raw_offset=offset,
            clip_len=int(dc["clip_len"]),
            window_stride=int(cal_cfg["inference"]["window_stride"]),
            batch_size=int(cal_cfg["inference"]["batch_size"]),
            device=device,
        )
        print("  decoded 10-Hz samples:", len(table))
        tables.append(table)

    features = pd.concat(tables, ignore_index=True)
    features.to_csv(FEATURES_PATH, index=False)
    print("Saved:", FEATURES_PATH)

# Confirm every sparse label has a dense prediction.
audit = labels[["ID", "sample_index"]].merge(
    features[["ID", "sample_index"]],
    on=["ID", "sample_index"],
    how="left",
    indicator=True,
)
assert audit["_merge"].eq("both").all(), audit[audit["_merge"] != "both"]

print("dense feature rows:", len(features))
print("labeled rows      :", len(labels))
display(features.head())


## 4. LOVO calibration

Acceleration and steering are calibrated separately because the official Stage 3
score is a weighted sum of their Macro-F1 values. Steering is evaluated with the
**ground-truth STOPPED mask**, exactly as in `metrics.py`.

The acceleration search is two-stage for speed:

1. scalar/fused source + STOP/deadzones/bias/smoothing,
2. only the best scalar candidates are expanded with ordinal blending.

This keeps the tiny public set from turning into an unnecessarily huge brute-force
search.


In [ ]:
cc = cal_cfg["calibration"]

report = leave_one_video_out_calibration(
    features,
    labels,
    ordinal_thresholds_mps2=thresholds,
    accel_grid=cc["accel_grid"],
    steer_grid=cc["steer_grid"],
    top_k_scalar=int(cc.get("top_k_scalar", 24)),
)

paths = save_calibration_report(report, CAL_OUTPUT_DIR)

print("LOVO mean:", json.dumps(report["cv_mean"], indent=2))
print("LOVO std :", json.dumps(report["cv_std"], indent=2))
print("\nConsensus on all public labels:")
print(json.dumps(report["consensus"]["metrics_on_all_labels"], indent=2))
print("\nAll-label best on all public labels:")
print(json.dumps(report["allfit"]["metrics_on_all_labels"], indent=2))

folds = pd.DataFrame(report["folds"])
display(folds)

print("\nConsensus calibration:")
print(json.dumps(report["consensus"]["calibration"], indent=2))
print("\nAll-fit calibration:")
print(json.dumps(report["allfit"]["calibration"], indent=2))

print("\nSaved:")
for name, path in paths.items():
    print(f"  {name:16s}: {path}")


## 5. Sanity comparison and first-submission choice

The public labels are **not** the hidden evaluation distribution, so the notebook
does not silently equate the all-label optimum with the best leaderboard choice.

Use these diagnostics:

- LOVO mean / fold variance,
- whether fold-selected parameters agree,
- consensus vs all-fit predictions,
- per-class confusion.

The default first submission uses `stage3_calibration_consensus.json`. Keep
`stage3_calibration_allfit.json` as a deliberate alternate.


In [ ]:
from sklearn.metrics import confusion_matrix

consensus = Stage3Calibration.from_dict(report["consensus"]["calibration"])
allfit = Stage3Calibration.from_dict(report["allfit"]["calibration"])

def merged_predictions(cal):
    pred = apply_calibration(features, cal)
    return labels.merge(
        pred,
        on=["ID", "sample_index"],
        how="left",
        validate="one_to_one",
        suffixes=("_true", "_pred"),
    )

for name, cal in [("CONSENSUS", consensus), ("ALLFIT", allfit)]:
    merged = merged_predictions(cal)
    metrics = dacon_stage3_metrics(
        merged["accel_label_true"],
        merged["accel_label_pred"],
        merged["steer_label_true"],
        merged["steer_label_pred"],
    )
    print("\n==", name, "==")
    print(metrics)

    accel_cm = confusion_matrix(
        merged["accel_label_true"],
        merged["accel_label_pred"],
        labels=["ACCELERATING", "DECELERATING", "CONSTANT", "STOPPED"],
    )
    display(pd.DataFrame(
        accel_cm,
        index=["GT_ACC", "GT_DEC", "GT_CONST", "GT_STOP"],
        columns=["P_ACC", "P_DEC", "P_CONST", "P_STOP"],
    ))

    moving = merged["accel_label_true"] != "STOPPED"
    steer_cm = confusion_matrix(
        merged.loc[moving, "steer_label_true"],
        merged.loc[moving, "steer_label_pred"],
        labels=["LEFT", "STRAIGHT", "RIGHT"],
    )
    display(pd.DataFrame(
        steer_cm,
        index=["GT_LEFT", "GT_STRAIGHT", "GT_RIGHT"],
        columns=["P_LEFT", "P_STRAIGHT", "P_RIGHT"],
    ))

DEFAULT_CAL_PATH = paths["consensus"]
print("\nFIRST SUBMISSION CALIBRATION:", DEFAULT_CAL_PATH)


## 6. Export a Stage 3 bundle staging folder

This does **not** build the final three-stage `submit.zip` yet. It stages the
files that the submission-oriented `predict_stage3_v4a` expects. The pinned
V-JEPA source tree is intentionally not copied here automatically because it is a
repository directory; we will package it once with the final Stage 1/2/3
submission bundle.


In [ ]:
BUNDLE_DIR = CAL_OUTPUT_DIR / "bundle_stage3"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(V4_CFG_PATH, BUNDLE_DIR / "vjepa21b_can_accel_v4a.yaml")
shutil.copy2(stats_path, BUNDLE_DIR / "target_stats.json")
shutil.copy2(V4_CKPT_DRIVE, BUNDLE_DIR / "v4a_best.pt")
shutil.copy2(VJEPA_DRIVE, BUNDLE_DIR / VJEPA_NAME)
shutil.copy2(DEFAULT_CAL_PATH, BUNDLE_DIR / "stage3_calibration.json")

print("Stage 3 bundle staging directory:", BUNDLE_DIR)
for p in sorted(BUNDLE_DIR.iterdir()):
    print(f"{p.name:40s} {p.stat().st_size / 2**20:10.1f} MiB")

print("\nStill required for final offline submit bundle:")
print("  - pinned vjepa2 source tree")
print("  - blackbox_detection package / inference integration")
print("  - existing Stage 1 and Stage 2 models")
print("  - top-level inference.py and DACON requirements.txt")


### Next step

After this notebook finishes, send:

- `calibration_report.json`
- `lovo_folds.csv`

The next notebook/code step is to package the selected calibration into the
existing DACON `inference.py`, run the released baseline smoke inputs end-to-end,
measure runtime, and produce the first real submission ZIP.
